# British Airways Flight Delay Analysis ? 01. Data Cleaning Pipeline
**Project:** End-to-End Operational Analytics & BI Dashboard  
**Phase:** Data Hygiene, Deduplication, Type Normalization & Quality Assurance  
**Author:** Senior Data Analyst & Airline Operations Team  

---
### Notebook Objectives:
1. Audit the raw flight operations dataset for structural anomalies, missing values, and duplication.
2. Standardize schema definitions, string encodings, and airport IATA identifiers.
3. Parse and validate flight datetime stamps, calculating scheduled and actual durations.
4. Correctly preserve operationally meaningful **negative delays** (early pushbacks and touch-downs).
5. Impute delay cause sub-components using standard aviation business rules.
6. Export the validated cleaned dataset for downstream feature engineering and SQL ingestion.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from data_cleaning import clean_flight_data, save_cleaned_data

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
print("Environment and libraries loaded successfully.")


Environment and libraries loaded successfully.


## 1. Raw Dataset Ingestion & Inspection

In [2]:
raw_path = os.path.join('..', 'data', 'raw', 'flight_data.csv')
if not os.path.exists(raw_path):
    raw_path = 'data/raw/flight_data.csv'

df_raw = pd.read_csv(raw_path)
print(f"Raw Dataset Loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head(5)


Raw Dataset Loaded: 75,681 rows, 24 columns


,flight_number,flight_date,airline,origin_airport,origin_city,dest_airport,dest_city,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,departure_delay,arrival_delay,distance_km,tail_number,aircraft_type,is_cancelled,cancellation_reason,is_diverted,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,BA8702,2023-07-28,British Airways,FRA,Frankfurt,LCY,London City,2023-07-28 06:20,2023-07-28 06:22,2023-07-28 07:44,2023-07-28 07:46,2.0,2.0,635,G-LCYD,Embraer E190,0,NaN,0,0.0,0.0,0.0,0.0,0.0
1,BA14306,2023-04-18,British Airways,EDI,Edinburgh,LHR,London Heathrow,2023-04-18 16:05,2023-04-18 16:16,2023-04-18 17:21,2023-04-18 17:56,11.0,35.0,534,G-TTNB,A320neo,0,NaN,0,26.6,0.0,0.0,0.0,8.4
2,BA5404,2023-07-28,British Airways,FCO,Rome Fiumicino,LHR,London Heathrow,2023-07-28 13:25,2023-07-28 13:35,2023-07-28 15:51,2023-07-28 15:55,10.0,4.0,1445,G-TTNA,A320neo,0,NaN,0,0.0,0.0,0.0,0.0,0.0
3,BA1004,2023-03-21,British Airways,DXB,Dubai Intl,LHR,London Heathrow,2023-03-21 21:25,2023-03-21 21:24,2023-03-22 05:02,2023-03-22 05:10,-1.0,8.0,5480,G-XWBD,A350-1000,0,NaN,0,0.0,0.0,0.0,0.0,0.0
4,BA4503,2023-06-06,British Airways,LHR,London Heathrow,MAD,Madrid Barajas,2023-06-06 13:10,2023-06-06 14:30,2023-06-06 15:21,2023-06-06 16:34,80.0,73.0,1246,G-EUUE,A320-200,0,NaN,0,24.1,0.0,0.0,0.0,48.9


In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75681 entries, 0 to 75680
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   flight_number        75681 non-null  object 
 1   flight_date          75681 non-null  object 
 2   airline              75681 non-null  object 
 3   origin_airport       75681 non-null  object 
 4   origin_city          75681 non-null  object 
 5   dest_airport         75681 non-null  object 
 6   dest_city            75681 non-null  object 
 7   scheduled_departure  75681 non-null  object 
 8   actual_departure     73754 non-null  object 
 9   scheduled_arrival    75681 non-null  object 
 10  actual_arrival       73600 non-null  object 
 11  departure_delay      73754 non-null  float64
 12  arrival_delay        73600 non-null  float64
 13  distance_km          75681 non-null  int64  
 14  tail_number          75681 non-null  object 
 15  aircraft_type        75681 non-null 

## 2. Duplicate Detection & Deduplication

In [4]:
dup_count = df_raw.duplicated().sum()
print(f"Total Exact Duplicate Rows Detected: {dup_count:,} ({dup_count/len(df_raw)*100:.2f}%)")

df_dedup = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Records remaining after deduplication: {len(df_dedup):,}")


Total Exact Duplicate Rows Detected: 601 (0.79%)


Records remaining after deduplication: 75,080


## 3. Missing Value Audit & Aviation Imputation Logic
In flight telemetry:
- Cancelled flights do not have actual arrival/departure timestamps.
- Non-delayed flights have `0.0` delay cause breakdowns (carrier, weather, NAS, etc.).


In [5]:
missing_summary = pd.DataFrame({
    'Missing_Count': df_dedup.isna().sum(),
    'Missing_Pct': (df_dedup.isna().sum() / len(df_dedup) * 100).round(2)
})
missing_summary[missing_summary['Missing_Count'] > 0]


,Missing_Count,Missing_Pct
actual_departure,1915,2.55
actual_arrival,2068,2.75
departure_delay,1915,2.55
arrival_delay,2068,2.75
cancellation_reason,73165,97.45
carrier_delay,1915,2.55
weather_delay,1915,2.55
nas_delay,1915,2.55
security_delay,1915,2.55
late_aircraft_delay,1915,2.55


## 4. Negative Delay Preservation (Early Flights)
Negative delays represent early departures/arrivals and indicate high ramp and turnaround efficiency. They must NOT be removed or set to 0.


In [6]:
early_dep = (df_dedup['departure_delay'] < 0).sum()
early_arr = (df_dedup['arrival_delay'] < 0).sum()
operated_count = (df_dedup['is_cancelled'] == 0).sum()

print(f"Early Departures (<0 min): {early_dep:,} ({early_dep/operated_count*100:.2f}% of operated flights)")
print(f"Early Arrivals (<0 min): {early_arr:,} ({early_arr/operated_count*100:.2f}% of operated flights)")
print(f"Minimum Departure Delay: {df_dedup['departure_delay'].min()} minutes")
print(f"Minimum Arrival Delay: {df_dedup['arrival_delay'].min()} minutes")


Early Departures (<0 min): 17,442 (23.84% of operated flights)
Early Arrivals (<0 min): 16,214 (22.16% of operated flights)
Minimum Departure Delay: -20.0 minutes
Minimum Arrival Delay: -27.0 minutes


## 5. End-to-End Data Cleaning Execution

In [7]:
cleaned_df, audit_log = clean_flight_data(df_raw, verbose=True)
save_cleaned_data(cleaned_df, output_path=os.path.join('..', 'data', 'processed', 'cleaned_flight_data.csv'))
print("Data Cleaning Pipeline Successfully Validated and Exported.")


--- DATA CLEANING PIPELINE STARTED ---
Initial Records: 75,681 | Initial Columns: 24
Duplicate Rows Identified: 601


Final Cleaned Records: 75,080 | Final Columns: 27
Date Range: 2023-01-01 to 2023-12-31
Unique Airports: 42 | Unique Aircraft Types: 12
--- DATA CLEANING PIPELINE COMPLETED ---


Cleaned dataset saved successfully to ..\data\processed\cleaned_flight_data.csv
Data Cleaning Pipeline Successfully Validated and Exported.
